In [43]:
checkpoint = "prajjwal1/bert-tiny"
tokenizer_checkpoint = "bert-base-uncased"
dataset_name = "imdb"

from transformers import AutoModel

model = AutoModel.from_pretrained(checkpoint)

from chop.tools import get_tokenized_dataset

dataset, tokenizer = get_tokenized_dataset(
    dataset=dataset_name,
    checkpoint=tokenizer_checkpoint,
    return_tokenizer=True,
)

from pathlib import Path
import dill

with open(f"{Path.home()}/tutorial_5_best_model.pkl", "rb") as f:
    base_model = dill.load(f)

INFO     Tokenizing dataset imdb with AutoTokenizer for bert-base-uncased.


# Task 1

In Tutorial 6, all layers allocated to IntegerLinear are allocated the same width and fractional width. This is suboptimal, as different layers may have different sensitivities to quantization.

-    Modify the code to allow different layers to have widths in the range [8, 16, 32] and fractional widths in the range [2, 4, 8]. Expose this choice as an additional hyperparameter for the Optuna sampler.

-    Run the search again, and plot a figure that has the number of trials on the x axis, and the maximum achieved accuracy up to that point on the y axis.



## Extended Precision Search

This notebook extends the original search to include **all supported precision types** for Linear layers in MASE:

### Supported Precision Types:
1. **float32** - Full precision (baseline)
2. **integer** - Fixed-point integer quantization with configurable width and fractional bits
3. **minifloat_denorm** - Minifloat with denormalized representation
4. **minifloat_ieee** - IEEE-compliant minifloat format
5. **log** - Logarithmic quantization
6. **block_fp** - Block floating point with shared exponents
7. **block_minifloat** - Block minifloat quantization
8. **block_log** - Block logarithmic quantization
9. **binary** - Binary neural networks (1-bit weights)
10. **binary_scaling** - Binary with scaling factors

### Search Space Parameters:
- **Integer**: width ∈ {8, 16, 32}, frac_width ∈ {2, 4, 8}
- **Minifloat**: width ∈ {8, 16}, exponent_width ∈ {3, 4, 5}, exponent_bias ∈ {7, 15}
- **Log**: width ∈ {4, 8}
- **Block-based**: block_size ∈ {8, 16, 32}
- **Binary**: stochastic ∈ {True, False}, bipolar ∈ {True, False}

### Methodology:
1. Run separate Optuna studies for each precision type
2. Each study optimizes layer-specific hyperparameters
3. Compare cumulative maximum accuracy across trials
4. Visualize and analyze results

In [44]:
# Defining Search Space with ALL Supported Precisions

import torch
from chop.nn.quantized.modules.linear import (
    LinearInteger,
    LinearMinifloatDenorm,
    LinearMinifloatIEEE,
    LinearLog,
    LinearBlockFP,
    LinearBlockMinifloat,
    LinearBlockLog,
    LinearBinary,
    LinearBinaryScaling,
)

# ============================================================================
# PRECISION TYPE DEFINITIONS
# Each precision type has a name, class, and configuration generator function
# ============================================================================

PRECISION_TYPES = {
    "float32": {
        "class": torch.nn.Linear,
        "config_generator": None,  # No config needed for full precision
    },
    "integer": {
        "class": LinearInteger,
        "config_generator": lambda trial, name, search_space: {
            "data_in_width": trial.suggest_categorical(f"{name}_int_width", search_space["int_width"]),
            "data_in_frac_width": trial.suggest_categorical(f"{name}_int_frac_width", search_space["frac_width"]),
            "weight_width": trial.suggest_categorical(f"{name}_weight_width", search_space["int_width"]),
            "weight_frac_width": trial.suggest_categorical(f"{name}_weight_frac_width", search_space["frac_width"]),
            "bias_width": trial.suggest_categorical(f"{name}_bias_width", search_space["int_width"]),
            "bias_frac_width": trial.suggest_categorical(f"{name}_bias_frac_width", search_space["frac_width"]),
        },
    },
    "minifloat_denorm": {
        "class": LinearMinifloatDenorm,
        "config_generator": lambda trial, name, search_space: {
            "data_in_width": trial.suggest_categorical(f"{name}_mfd_width", search_space["minifloat_width"]),
            "data_in_exponent_width": trial.suggest_categorical(f"{name}_mfd_exp_width", search_space["exponent_width"]),
            "data_in_exponent_bias": trial.suggest_categorical(f"{name}_mfd_exp_bias", search_space["exponent_bias"]),
            "weight_width": trial.suggest_categorical(f"{name}_mfd_w_width", search_space["minifloat_width"]),
            "weight_exponent_width": trial.suggest_categorical(f"{name}_mfd_w_exp_width", search_space["exponent_width"]),
            "weight_exponent_bias": trial.suggest_categorical(f"{name}_mfd_w_exp_bias", search_space["exponent_bias"]),
            "bias_width": trial.suggest_categorical(f"{name}_mfd_b_width", search_space["minifloat_width"]),
            "bias_exponent_width": trial.suggest_categorical(f"{name}_mfd_b_exp_width", search_space["exponent_width"]),
            "bias_exponent_bias": trial.suggest_categorical(f"{name}_mfd_b_exp_bias", search_space["exponent_bias"]),
        },
    },
    "minifloat_ieee": {
        "class": LinearMinifloatIEEE,
        "config_generator": lambda trial, name, search_space: {
            "data_in_width": trial.suggest_categorical(f"{name}_mfi_width", search_space["minifloat_width"]),
            "data_in_exponent_width": trial.suggest_categorical(f"{name}_mfi_exp_width", search_space["exponent_width"]),
            "data_in_exponent_bias": trial.suggest_categorical(f"{name}_mfi_exp_bias", search_space["exponent_bias"]),
            "weight_width": trial.suggest_categorical(f"{name}_mfi_w_width", search_space["minifloat_width"]),
            "weight_exponent_width": trial.suggest_categorical(f"{name}_mfi_w_exp_width", search_space["exponent_width"]),
            "weight_exponent_bias": trial.suggest_categorical(f"{name}_mfi_w_exp_bias", search_space["exponent_bias"]),
            "bias_width": trial.suggest_categorical(f"{name}_mfi_b_width", search_space["minifloat_width"]),
            "bias_exponent_width": trial.suggest_categorical(f"{name}_mfi_b_exp_width", search_space["exponent_width"]),
            "bias_exponent_bias": trial.suggest_categorical(f"{name}_mfi_b_exp_bias", search_space["exponent_bias"]),
        },
    },
    "log": {
        "class": LinearLog,
        "config_generator": lambda trial, name, search_space: {
            "data_in_width": trial.suggest_categorical(f"{name}_log_width", search_space["log_width"]),
            "data_in_exponent_bias": trial.suggest_categorical(f"{name}_log_exp_bias", search_space["exponent_bias"]),
            "weight_width": trial.suggest_categorical(f"{name}_log_w_width", search_space["log_width"]),
            "weight_exponent_bias": trial.suggest_categorical(f"{name}_log_w_exp_bias", search_space["exponent_bias"]),
            "bias_width": trial.suggest_categorical(f"{name}_log_b_width", search_space["log_width"]),
            "bias_exponent_bias": trial.suggest_categorical(f"{name}_log_b_exp_bias", search_space["exponent_bias"]),
        },
    },
    "block_fp": {
        "class": LinearBlockFP,
        "config_generator": lambda trial, name, search_space: {
            "data_in_width": trial.suggest_categorical(f"{name}_bfp_width", search_space["minifloat_width"]),
            "data_in_exponent_width": trial.suggest_categorical(f"{name}_bfp_exp_width", search_space["exponent_width"]),
            "data_in_exponent_bias": trial.suggest_categorical(f"{name}_bfp_exp_bias", search_space["exponent_bias"]),
            "data_in_block_size": trial.suggest_categorical(f"{name}_bfp_block_size", search_space["block_size"]),
            "weight_width": trial.suggest_categorical(f"{name}_bfp_w_width", search_space["minifloat_width"]),
            "weight_exponent_width": trial.suggest_categorical(f"{name}_bfp_w_exp_width", search_space["exponent_width"]),
            "weight_exponent_bias": trial.suggest_categorical(f"{name}_bfp_w_exp_bias", search_space["exponent_bias"]),
            "weight_block_size": trial.suggest_categorical(f"{name}_bfp_w_block_size", search_space["block_size"]),
            "bias_width": trial.suggest_categorical(f"{name}_bfp_b_width", search_space["minifloat_width"]),
            "bias_exponent_width": trial.suggest_categorical(f"{name}_bfp_b_exp_width", search_space["exponent_width"]),
            "bias_exponent_bias": trial.suggest_categorical(f"{name}_bfp_b_exp_bias", search_space["exponent_bias"]),
            "bias_block_size": trial.suggest_categorical(f"{name}_bfp_b_block_size", search_space["block_size"]),
        },
    },
    "block_minifloat": {
        "class": LinearBlockMinifloat,
        "config_generator": lambda trial, name, search_space: {
            "data_in_width": trial.suggest_categorical(f"{name}_bmf_width", search_space["minifloat_width"]),
            "data_in_exponent_width": trial.suggest_categorical(f"{name}_bmf_exp_width", search_space["exponent_width"]),
            "data_in_exponent_bias_width": trial.suggest_categorical(f"{name}_bmf_exp_bias_width", search_space["exponent_bias_width"]),
            "data_in_block_size": trial.suggest_categorical(f"{name}_bmf_block_size", search_space["block_size"]),
            "weight_width": trial.suggest_categorical(f"{name}_bmf_w_width", search_space["minifloat_width"]),
            "weight_exponent_width": trial.suggest_categorical(f"{name}_bmf_w_exp_width", search_space["exponent_width"]),
            "weight_exponent_bias_width": trial.suggest_categorical(f"{name}_bmf_w_exp_bias_width", search_space["exponent_bias_width"]),
            "weight_block_size": trial.suggest_categorical(f"{name}_bmf_w_block_size", search_space["block_size"]),
            "bias_width": trial.suggest_categorical(f"{name}_bmf_b_width", search_space["minifloat_width"]),
            "bias_exponent_width": trial.suggest_categorical(f"{name}_bmf_b_exp_width", search_space["exponent_width"]),
            "bias_exponent_bias_width": trial.suggest_categorical(f"{name}_bmf_b_exp_bias_width", search_space["exponent_bias_width"]),
            "bias_block_size": trial.suggest_categorical(f"{name}_bmf_b_block_size", search_space["block_size"]),
        },
    },
    "block_log": {
        "class": LinearBlockLog,
        "config_generator": lambda trial, name, search_space: {
            "data_in_width": trial.suggest_categorical(f"{name}_blog_width", search_space["log_width"]),
            "data_in_exponent_bias_width": trial.suggest_categorical(f"{name}_blog_exp_bias_width", search_space["exponent_bias_width"]),
            "data_in_block_size": trial.suggest_categorical(f"{name}_blog_block_size", search_space["block_size"]),
            "weight_width": trial.suggest_categorical(f"{name}_blog_w_width", search_space["log_width"]),
            "weight_exponent_bias_width": trial.suggest_categorical(f"{name}_blog_w_exp_bias_width", search_space["exponent_bias_width"]),
            "weight_block_size": trial.suggest_categorical(f"{name}_blog_w_block_size", search_space["block_size"]),
            "bias_width": trial.suggest_categorical(f"{name}_blog_b_width", search_space["log_width"]),
            "bias_exponent_bias_width": trial.suggest_categorical(f"{name}_blog_b_exp_bias_width", search_space["exponent_bias_width"]),
            "bias_block_size": trial.suggest_categorical(f"{name}_blog_b_block_size", search_space["block_size"]),
        },
    },
    "binary": {
        "class": LinearBinary,
        "config_generator": lambda trial, name, search_space: {
            "weight_stochastic": trial.suggest_categorical(f"{name}_bin_w_stochastic", [True, False]),
            "weight_bipolar": trial.suggest_categorical(f"{name}_bin_w_bipolar", [True, False]),
        },
    },
    "binary_scaling": {
        "class": LinearBinaryScaling,
        "config_generator": lambda trial, name, search_space: {
            "data_in_stochastic": trial.suggest_categorical(f"{name}_bins_x_stochastic", [True, False]),
            "data_in_bipolar": trial.suggest_categorical(f"{name}_bins_x_bipolar", [True, False]),
            "weight_stochastic": trial.suggest_categorical(f"{name}_bins_w_stochastic", [True, False]),
            "weight_bipolar": trial.suggest_categorical(f"{name}_bins_w_bipolar", [True, False]),
            "bias_stochastic": trial.suggest_categorical(f"{name}_bins_b_stochastic", [True, False]),
            "bias_bipolar": trial.suggest_categorical(f"{name}_bins_b_bipolar", [True, False]),
            "binary_training": True,  # Always enable binary training for search
        },
    },
}

# ============================================================================
# SEARCH SPACE DEFINITION
# Defines the hyperparameter ranges for each precision type
# ============================================================================

search_space = {
    # Integer quantization parameters
    "int_width": [8, 16, 32],
    "frac_width": [2, 4, 8],
    
    # Minifloat parameters (for denorm, IEEE, block_fp, block_minifloat)
    "minifloat_width": [8, 16],
    "exponent_width": [3, 4, 5],
    "exponent_bias": [7, 15],
    
    # Log quantization parameters
    "log_width": [4, 8],
    
    # Block-based quantization parameters
    "block_size": [8, 16, 32],
    "exponent_bias_width": [4, 8],
    
    # Available precision types for layer selection
    "precision_types": list(PRECISION_TYPES.keys()),
}

# ============================================================================
# MODEL CONSTRUCTOR
# Creates a model with layers quantized according to Optuna suggestions
# ============================================================================

from chop.tools.utils import deepsetattr
from copy import deepcopy
param_list = []

def construct_model(trial, precision_type=None):
    """
    Construct a model with quantized layers based on Optuna trial suggestions.
    
    Args:
        trial: Optuna trial object for hyperparameter suggestions
        precision_type: If specified, use this precision type for all layers.
                       If None, let Optuna choose per-layer precision.
    
    Returns:
        A copy of base_model with layers replaced according to precision choices
    """
    trial_model = deepcopy(base_model)
    
    for name, layer in trial_model.named_modules():
        if isinstance(layer, torch.nn.Linear):
            # Determine the precision type for this layer
            if precision_type is not None:
                # Use the specified precision type for all layers
                chosen_precision = precision_type
            else:
                # Let Optuna choose the precision type per layer
                chosen_precision = trial.suggest_categorical(
                    f"{name}_precision_type",
                    search_space["precision_types"],
                )
            
            precision_info = PRECISION_TYPES[chosen_precision]
            new_layer_cls = precision_info["class"]
            
            # Skip if full precision (torch.nn.Linear)
            if new_layer_cls == torch.nn.Linear:
                continue
            
            # Build keyword arguments for the new layer
            kwargs = {
                "in_features": layer.in_features,
                "out_features": layer.out_features,
                "bias": layer.bias is not None,
            }
            
            # Generate config using the precision-specific config generator
            config_generator = precision_info["config_generator"]
            if config_generator is not None:
                kwargs["config"] = config_generator(trial, name, search_space)
            
            # Create the new layer and copy weights
            new_layer = new_layer_cls(**kwargs)
            new_layer.weight.data = layer.weight.data.clone()
            if layer.bias is not None and new_layer.bias is not None:
                new_layer.bias.data = layer.bias.data.clone()
            
            # Replace the layer in the model
            deepsetattr(trial_model, name, new_layer)

    param_list.append(f"Trial params: {trial.params}")
    return trial_model


def get_precision_class_from_module(module):
    """
    Get the precision type name from a module instance.
    """
    for precision_name, precision_info in PRECISION_TYPES.items():
        if isinstance(module, precision_info["class"]):
            return precision_name
    return "unknown"


print("✓ Search space defined with", len(PRECISION_TYPES), "precision types:")
for name in PRECISION_TYPES:
    print(f"  - {name}: {PRECISION_TYPES[name]['class'].__name__}")

✓ Search space defined with 10 precision types:
  - float32: Linear
  - integer: LinearInteger
  - minifloat_denorm: LinearMinifloatDenorm
  - minifloat_ieee: LinearMinifloatIEEE
  - log: LinearLog
  - block_fp: LinearBlockFP
  - block_minifloat: LinearBlockMinifloat
  - block_log: LinearBlockLog
  - binary: LinearBinary
  - binary_scaling: LinearBinaryScaling


In [45]:
# Print search space summary
print("=" * 60)
print("SEARCH SPACE SUMMARY")
print("=" * 60)
for key, values in search_space.items():
    print(f"{key}: {values}")

SEARCH SPACE SUMMARY
int_width: [8, 16, 32]
frac_width: [2, 4, 8]
minifloat_width: [8, 16]
exponent_width: [3, 4, 5]
exponent_bias: [7, 15]
log_width: [4, 8]
block_size: [8, 16, 32]
exponent_bias_width: [4, 8]
precision_types: ['float32', 'integer', 'minifloat_denorm', 'minifloat_ieee', 'log', 'block_fp', 'block_minifloat', 'block_log', 'binary', 'binary_scaling']


In [46]:
# ============================================================================
# STUDY CONFIGURATION
# Configuration for running precision-specific studies
# ============================================================================

from optuna.samplers import GridSampler, RandomSampler, TPESampler

# Configuration
N_TRIALS_PER_PRECISION = 5  # Number of trials per precision type
SAMPLER_TYPE = "random"  # Options: "random", "tpe", "grid"

# Select the sampler
if SAMPLER_TYPE == "tpe":
    sampler_factory = lambda: TPESampler()
elif SAMPLER_TYPE == "grid":
    sampler_factory = lambda: GridSampler(search_space)  # Note: Grid may not work well with large spaces
else:
    sampler_factory = lambda: RandomSampler()

print(f"Using {SAMPLER_TYPE.upper()} sampler")
print(f"Running {N_TRIALS_PER_PRECISION} trials per precision type")
print(f"Total precision types: {len(PRECISION_TYPES)}")

Using RANDOM sampler
Running 5 trials per precision type
Total precision types: 10


In [47]:
# ============================================================================
# OBJECTIVE FUNCTION FACTORY
# Creates precision-specific objective functions for Optuna
# ============================================================================

from chop.tools import get_trainer
import traceback


def create_objective(precision_type):
    """
    Create an objective function for a specific precision type.
    
    Args:
        precision_type: The precision type to use for all layers in this study
        
    Returns:
        An objective function for Optuna optimization
    """
    def objective(trial):
        try:
            # Construct model with the specified precision type
            model = construct_model(trial, precision_type=precision_type)
            
            # Create trainer
            trainer = get_trainer(
                model=model,
                tokenized_dataset=dataset,
                tokenizer=tokenizer,
                evaluate_metric="accuracy",
                num_train_epochs=1,
            )
            
            # Train and evaluate
            trainer.train()
            eval_results = trainer.evaluate()
            
            # Store the model in trial for later retrieval
            trial.set_user_attr("model", model)
            trial.set_user_attr("precision_type", precision_type)
            
            return eval_results["eval_accuracy"]
            
        except Exception as e:
            print(f"Trial failed for {precision_type}: {str(e)}")
            traceback.print_exc()
            # Return a very low accuracy for failed trials
            return 0.0
    
    return objective


print("✓ Objective function factory created")

✓ Objective function factory created


In [48]:
# ============================================================================
# RUN PRECISION-SPECIFIC STUDIES
# Run a separate Optuna study for each precision type
# ============================================================================

import optuna
from collections import defaultdict
import warnings

# Suppress Optuna's informational messages for cleaner output
optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings("ignore", category=UserWarning)

# Store results for each precision type
precision_results = {}

# Run a study for each precision type
for precision_name in PRECISION_TYPES.keys():
    print(f"\n{'='*60}")
    print(f"Running study for precision: {precision_name.upper()}")
    print(f"{'='*60}")
    
    # Create study for this precision type
    study = optuna.create_study(
        direction="maximize",
        study_name=f"bert-tiny-{precision_name}-study",
        sampler=sampler_factory(),
    )
    
    # Create objective function for this precision
    objective_fn = create_objective(precision_name)
    
    # Run optimization
    try:
        study.optimize(
            objective_fn,
            n_trials=N_TRIALS_PER_PRECISION,
            timeout=60 * 60,  # 1 hour timeout per precision
            show_progress_bar=True,
        )
        
        # Store results
        precision_results[precision_name] = {
            "study": study,
            "trials": study.trials,
            "best_trial": study.best_trial if len(study.trials) > 0 else None,
            "best_value": study.best_value if len(study.trials) > 0 else 0.0,
            "trial_values": [t.value if t.value is not None else 0.0 for t in study.trials],
        }
        
        print(f"✓ {precision_name}: Best accuracy = {precision_results[precision_name]['best_value']:.4f}")
        
    except Exception as e:
        print(f"✗ Study for {precision_name} failed: {str(e)}")
        precision_results[precision_name] = {
            "study": None,
            "trials": [],
            "best_trial": None,
            "best_value": 0.0,
            "trial_values": [],
        }

print(f"\n{'='*60}")
print("ALL STUDIES COMPLETED")
print(f"{'='*60}")


Running study for precision: FLOAT32


  0%|          | 0/5 [00:00<?, ?it/s]/home/ism/ADL/mase/src/chop/tools/huggingface.py:157: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss
500,0.368900
1000,0.324600
1500,0.321100
2000,0.344700
2500,0.333300
3000,0.371600


Best trial: 0. Best value: 0.86692:  20%|██        | 1/5 [01:10<04:40, 70.01s/it, 70.00/3600 seconds]/home/ism/ADL/mase/src/chop/tools/huggingface.py:157: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss
500,0.368900
1000,0.324600
1500,0.321100
2000,0.344700
2500,0.333300
3000,0.371600


Best trial: 0. Best value: 0.86692:  40%|████      | 2/5 [02:18<03:27, 69.02s/it, 138.33/3600 seconds]/home/ism/ADL/mase/src/chop/tools/huggingface.py:157: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss
500,0.368900
1000,0.324600


Best trial: 0. Best value: 0.86692:  40%|████      | 2/5 [02:42<04:04, 81.34s/it, 138.33/3600 seconds]


[W 2026-02-03 13:31:10,764] Trial 2 failed with parameters: {} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/home/ism/ADL/mase/.venv/lib/python3.11/site-packages/optuna/study/_optimize.py", line 205, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/tmp/ipykernel_396813/1530101190.py", line 35, in objective
    trainer.train()
  File "/home/ism/ADL/mase/.venv/lib/python3.11/site-packages/transformers/trainer.py", line 2245, in train
    return inner_training_loop(
           ^^^^^^^^^^^^^^^^^^^^
  File "/home/ism/ADL/mase/.venv/lib/python3.11/site-packages/transformers/trainer.py", line 2514, in _inner_training_loop
    batch_samples, num_items_in_batch = self.get_batch_samples(epoch_iterator, num_batches, args.device)
                                        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/ism/ADL/mase/.venv/lib/python3.11/site-packages/transformers

KeyboardInterrupt: 

In [49]:
print("Parameter list:", param_list)

Parameter list: ['Trial params: {}', 'Trial params: {}', 'Trial params: {}']


In [ ]:
# ============================================================================
# RESULTS ANALYSIS AND VISUALIZATION
# Plot comparison of precision types
# ============================================================================

import matplotlib.pyplot as plt
import numpy as np

def compute_cumulative_max(values):
    """Compute cumulative maximum accuracy up to each trial."""
    if len(values) == 0:
        return []
    cummax = []
    current_max = float('-inf')
    for v in values:
        current_max = max(current_max, v)
        cummax.append(current_max)
    return cummax


# Create figure with two subplots
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ============================================================================
# Plot 1: Cumulative Maximum Accuracy vs Trial Number
# ============================================================================
ax1 = axes[0]

# Color palette for different precisions
colors = plt.cm.tab10(np.linspace(0, 1, len(PRECISION_TYPES)))

for idx, (precision_name, results) in enumerate(precision_results.items()):
    trial_values = results["trial_values"]
    
    if len(trial_values) == 0:
        continue
    
    # Compute cumulative maximum
    cummax_values = compute_cumulative_max(trial_values)
    trial_numbers = list(range(1, len(cummax_values) + 1))
    
    # Plot cumulative max curve
    ax1.plot(
        trial_numbers, 
        cummax_values, 
        marker='o', 
        markersize=6,
        linewidth=2,
        label=f"{precision_name}",
        color=colors[idx],
        alpha=0.8,
    )

ax1.set_xlabel("Trial Number", fontsize=12)
ax1.set_ylabel("Maximum Achieved Accuracy", fontsize=12)
ax1.set_title("Cumulative Maximum Accuracy per Precision Type", fontsize=14, fontweight='bold')
ax1.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)
ax1.grid(True, alpha=0.3)
ax1.set_xlim(left=0.5)

# ============================================================================
# Plot 2: Best Accuracy Bar Chart Comparison
# ============================================================================
ax2 = axes[1]

# Prepare data for bar chart
precision_names = []
best_accuracies = []
bar_colors = []

for idx, (precision_name, results) in enumerate(precision_results.items()):
    precision_names.append(precision_name)
    best_accuracies.append(results["best_value"])
    bar_colors.append(colors[idx])

# Create bar chart
bars = ax2.bar(
    range(len(precision_names)), 
    best_accuracies, 
    color=bar_colors,
    edgecolor='black',
    linewidth=1,
    alpha=0.8,
)

# Add value labels on bars
for bar, acc in zip(bars, best_accuracies):
    height = bar.get_height()
    ax2.annotate(
        f'{acc:.3f}',
        xy=(bar.get_x() + bar.get_width() / 2, height),
        xytext=(0, 3),
        textcoords="offset points",
        ha='center', 
        va='bottom',
        fontsize=9,
        fontweight='bold',
    )

ax2.set_xticks(range(len(precision_names)))
ax2.set_xticklabels(precision_names, rotation=45, ha='right', fontsize=10)
ax2.set_ylabel("Best Accuracy", fontsize=12)
ax2.set_title("Best Accuracy by Precision Type", fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig("precision_comparison.png", dpi=150, bbox_inches='tight')
plt.show()

print("\n✓ Figure saved to 'precision_comparison.png'")

[I 2026-02-03 12:51:15,546] A new study created in memory with name: bert-tiny-nas-study


Layer name: bert.encoder.layer.0.attention.self.query
Layer bert.encoder.layer.0.attention.self.query - kwargs['config']: {'data_in_width': 8, 'data_in_frac_width': 2, 'weight_width': 8, 'weight_frac_width': 2, 'bias_width': 8, 'bias_frac_width': 2}
Layer name: bert.encoder.layer.0.attention.self.value
Layer bert.encoder.layer.0.attention.self.value - kwargs['config']: {'data_in_width': 8, 'data_in_frac_width': 8, 'weight_width': 8, 'weight_frac_width': 8, 'bias_width': 8, 'bias_frac_width': 8}
Layer name: bert.encoder.layer.1.output.dense
Layer bert.encoder.layer.1.output.dense - kwargs['config']: {'data_in_width': 16, 'data_in_frac_width': 2, 'weight_width': 16, 'weight_frac_width': 2, 'bias_width': 16, 'bias_frac_width': 2}
Layer name: classifier
Layer classifier - kwargs['config']: {'data_in_width': 16, 'data_in_frac_width': 4, 'weight_width': 16, 'weight_frac_width': 4, 'bias_width': 16, 'bias_frac_width': 4}


/home/ism/ADL/mase/.venv/lib/python3.11/site-packages/optuna/distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains <class 'torch.nn.modules.linear.Linear'> which is of type type.
  warnings.warn(message)
/home/ism/ADL/mase/.venv/lib/python3.11/site-packages/optuna/distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains <class 'chop.nn.quantized.modules.linear.LinearInteger'> which is of type type.
  warnings.warn(message)
/home/ism/ADL/mase/src/chop/tools/huggingface.py:157: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss
500,0.560200
1000,0.416000
1500,0.384100
2000,0.364100
2500,0.342900
3000,0.370800


[I 2026-02-03 12:52:41,826] Trial 0 finished with value: 0.85968 and parameters: {'bert.encoder.layer.0.attention.self.query_type': <class 'chop.nn.quantized.modules.linear.LinearInteger'>, 'bert.encoder.layer.0.attention.self.query_int_width': 0, 'bert.encoder.layer.0.attention.self.query_frac_width': 0, 'bert.encoder.layer.0.attention.self.value_type': <class 'chop.nn.quantized.modules.linear.LinearInteger'>, 'bert.encoder.layer.0.attention.self.value_int_width': 0, 'bert.encoder.layer.0.attention.self.value_frac_width': 2, 'bert.encoder.layer.0.intermediate.dense_type': <class 'torch.nn.modules.linear.Linear'>, 'bert.encoder.layer.0.output.dense_type': <class 'torch.nn.modules.linear.Linear'>, 'bert.encoder.layer.1.attention.output.dense_type': <class 'torch.nn.modules.linear.Linear'>, 'bert.encoder.layer.1.intermediate.dense_type': <class 'torch.nn.modules.linear.Linear'>, 'bert.encoder.layer.1.output.dense_type': <class 'chop.nn.quantized.modules.linear.LinearInteger'>, 'bert.enco

In [ ]:
# ============================================================================
# DETAILED RESULTS SUMMARY TABLE
# ============================================================================

import pandas as pd

# Create summary DataFrame
summary_data = []
for precision_name, results in precision_results.items():
    trial_values = results["trial_values"]
    summary_data.append({
        "Precision Type": precision_name,
        "Best Accuracy": f"{results['best_value']:.4f}" if results['best_value'] > 0 else "N/A",
        "Mean Accuracy": f"{np.mean(trial_values):.4f}" if len(trial_values) > 0 else "N/A",
        "Std Accuracy": f"{np.std(trial_values):.4f}" if len(trial_values) > 0 else "N/A",
        "Completed Trials": len([v for v in trial_values if v > 0]),
        "Total Trials": len(trial_values),
    })

summary_df = pd.DataFrame(summary_data)
summary_df = summary_df.sort_values("Best Accuracy", ascending=False)

print("=" * 80)
print("PRECISION TYPE PERFORMANCE SUMMARY")
print("=" * 80)
print(summary_df.to_string(index=False))
print("=" * 80)

In [ ]:
# ============================================================================
# BEST CONFIGURATION ANALYSIS
# Display the best hyperparameter configurations for each precision type
# ============================================================================

print("=" * 80)
print("BEST CONFIGURATIONS PER PRECISION TYPE")
print("=" * 80)

for precision_name, results in precision_results.items():
    best_trial = results.get("best_trial")
    
    if best_trial is None:
        print(f"\n{precision_name.upper()}: No successful trials")
        continue
    
    print(f"\n{precision_name.upper()}")
    print(f"  Best Accuracy: {results['best_value']:.4f}")
    print(f"  Best Trial #:  {best_trial.number}")
    print(f"  Parameters:")
    
    # Group parameters by layer
    params_by_layer = {}
    for param_name, param_value in best_trial.params.items():
        # Extract layer name from parameter name
        parts = param_name.rsplit('_', 1)
        if len(parts) == 2:
            layer_name = parts[0]
            param_key = parts[1]
        else:
            layer_name = "global"
            param_key = param_name
        
        if layer_name not in params_by_layer:
            params_by_layer[layer_name] = {}
        params_by_layer[layer_name][param_key] = param_value
    
    # Print first 3 layers' configurations
    for i, (layer_name, params) in enumerate(list(params_by_layer.items())[:3]):
        print(f"    {layer_name}: {params}")
    
    if len(params_by_layer) > 3:
        print(f"    ... and {len(params_by_layer) - 3} more layers")

print("\n" + "=" * 80)

In [ ]:
# ============================================================================
# ADDITIONAL VISUALIZATION: Box Plot of Accuracy Distribution
# ============================================================================

fig, ax = plt.subplots(figsize=(14, 6))

# Prepare data for box plot
box_data = []
box_labels = []
box_positions = []

for idx, (precision_name, results) in enumerate(precision_results.items()):
    trial_values = [v for v in results["trial_values"] if v > 0]  # Filter out failed trials
    if len(trial_values) > 0:
        box_data.append(trial_values)
        box_labels.append(precision_name)
        box_positions.append(idx)

if len(box_data) > 0:
    # Create box plot
    bp = ax.boxplot(
        box_data, 
        positions=box_positions,
        patch_artist=True,
        widths=0.6,
    )
    
    # Color the boxes
    for patch, color in zip(bp['boxes'], colors[:len(box_data)]):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    
    # Scatter individual points
    for idx, (data, pos) in enumerate(zip(box_data, box_positions)):
        x_jitter = np.random.normal(pos, 0.04, len(data))
        ax.scatter(x_jitter, data, alpha=0.6, s=30, color=colors[idx], zorder=3)
    
    ax.set_xticks(box_positions)
    ax.set_xticklabels(box_labels, rotation=45, ha='right', fontsize=10)
    ax.set_ylabel("Accuracy", fontsize=12)
    ax.set_title("Accuracy Distribution by Precision Type (Box Plot)", fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.savefig("precision_boxplot.png", dpi=150, bbox_inches='tight')
    plt.show()
    
    print("\n✓ Box plot saved to 'precision_boxplot.png'")
else:
    print("No successful trials to plot.")

In [ ]:
# ============================================================================
# SAVE RESULTS AND BEST MODEL
# ============================================================================

import json
from pathlib import Path

# Find the overall best precision type and model
best_precision = max(precision_results.keys(), key=lambda k: precision_results[k]["best_value"])
best_overall_accuracy = precision_results[best_precision]["best_value"]
best_trial_obj = precision_results[best_precision]["best_trial"]

print("=" * 80)
print("FINAL RESULTS")
print("=" * 80)
print(f"Best Precision Type: {best_precision}")
print(f"Best Overall Accuracy: {best_overall_accuracy:.4f}")
print("=" * 80)

# Export results to JSON
export_data = {
    "experiment_config": {
        "n_trials_per_precision": N_TRIALS_PER_PRECISION,
        "sampler_type": SAMPLER_TYPE,
        "precision_types": list(PRECISION_TYPES.keys()),
    },
    "results": {}
}

for precision_name, results in precision_results.items():
    export_data["results"][precision_name] = {
        "best_accuracy": results["best_value"],
        "trial_accuracies": results["trial_values"],
        "mean_accuracy": float(np.mean(results["trial_values"])) if results["trial_values"] else 0.0,
        "std_accuracy": float(np.std(results["trial_values"])) if results["trial_values"] else 0.0,
        "best_params": dict(results["best_trial"].params) if results["best_trial"] else None,
    }

# Save to JSON file
output_path = Path.home() / "precision_search_results.json"
with open(output_path, "w") as f:
    json.dump(export_data, f, indent=2)

print(f"\n✓ Results exported to: {output_path}")

# Save the best model using dill
if best_trial_obj is not None:
    best_model = best_trial_obj.user_attrs.get("model")
    if best_model is not None:
        model_path = Path.home() / f"best_model_{best_precision}.pkl"
        with open(model_path, "wb") as f:
            dill.dump(best_model, f)
        print(f"✓ Best model saved to: {model_path}")
    else:
        print("⚠ Best model not available in trial attributes")
else:
    print("⚠ No best trial available")